In [ ]:
# -*- coding: utf-8 -*-

import time
import random
import warnings
import sys
import atexit
import msvcrt
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import networkx as nx
import torch
import torch.nn.functional as F

import matplotlib.pyplot as plt
import matplotlib.image as mpimg
from matplotlib import font_manager

from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, f1_score

from torch_geometric.data import Data
from torch_geometric.utils import to_undirected, add_self_loops, to_networkx
from torch_geometric.nn import GCNConv

warnings.filterwarnings("ignore")


# =========================================================
# 0-A. ログ保存（画面表示 + txt保存 同時出力）
# =========================================================

class Tee:
    def __init__(self, *files):
        self.files = files

    def write(self, obj):
        for f in self.files:
            try:
                f.write(obj)
                f.flush()
            except Exception:
                pass

    def flush(self):
        for f in self.files:
            try:
                f.flush()
            except Exception:
                pass


LOG_FILE = f"execution_log_{datetime.now().strftime('%Y%m%d_%H%M%S')}.txt"
_original_stdout = sys.stdout
_original_stderr = sys.stderr
_log_fp = open(LOG_FILE, "w", encoding="utf-8")
sys.stdout = Tee(_original_stdout, _log_fp)
sys.stderr = Tee(_original_stderr, _log_fp)


def close_log_file():
    try:
        sys.stdout.flush()
        sys.stderr.flush()
        _log_fp.close()
    except Exception:
        pass


atexit.register(close_log_file)
print(f"ログ保存先: {LOG_FILE}", flush=True)



# =========================================================
# 0. 主要パラメータ
# =========================================================

TXS_FEATURES = "./transactions/txs_features.txt"
TXS_CLASSES  = "./transactions/txs_classes.txt"
TXS_EDGES    = "./transactions/txs_edgelist.txt"

SEED = 42

TRAIN_END_STEP = 34
VAL_END_STEP   = 39

EPOCHS       = 50
LR           = 0.001
WEIGHT_DECAY = 5e-4
HIDDEN_DIM   = 64
DROPOUT      = 0.5

LOG_EPOCH_INTERVAL = 5

BETWEENNESS_K = 100

HITS_MAX_ITER = 1000
HITS_TOL = 1.0e-8

# 例: [100] + list(range(5000, 150001, 5000))
PSEUDO_LIST = [100] + list(range(5000, 150001, 5000))

PSEUDO_ONLY_TRAIN_PERIOD = True

# 中心性 + 時間異常指標
METRIC_MODES = [
    "degree",
    "betweenness",
    "pagerank",
    "closeness",
    "hits_hub",
    "constraint",
    "temporal_burst",
    "transaction_velocity",
]

# 互換性のため、元コードと同じ名前も残す
CENTRALITY_MODES = METRIC_MODES

INPUT_TIMEOUT_SEC = 10
INPUT_DEFAULT = "y"

PLOT_FIGSIZE = (15, 8)
PLOT_DPI = 200


OUTPUT_CENTER_ALL = "unknown_outlier_center_all_rankings.csv"

# エッジランキング出力
ENABLE_EDGE_RANKING = True
EDGE_RANKING_FILE = "edge_ranking_all.csv"
EDGE_RANKING_TOP_FILE = "edge_ranking_top1000.csv"
EDGE_RANKING_TOP_N = 1000

# =========================================================
# 枝刈り設定
# =========================================================
ENABLE_EDGE_PRUNING = True
PRUNE_MODE = "ratio"      # "ratio" または "top_n"
PRUNE_KEEP_RATIO = 0.3     # 上位30%のedgeを残す
PRUNE_TOP_N = 100000       # PRUNE_MODE="top_n" のとき使用
PRUNE_KEEP_AT_LEAST_ONE_EDGE_PER_NODE = True
PRUNED_EDGE_FILE = "edge_pruned_selected_edges.csv"



# =========================================================
# 1. 日本語フォント設定
# =========================================================

def setup_japanese_font():
    candidates = [
        "Yu Gothic",
        "MS Gothic",
        "Meiryo",
        "Hiragino Sans",
        "IPAexGothic",
        "IPAPGothic",
    ]

    available = set(f.name for f in font_manager.fontManager.ttflist)

    for font in candidates:
        if font in available:
            plt.rcParams["font.family"] = font
            break

    plt.rcParams["axes.unicode_minus"] = False


# =========================================================
# 2. ログ・共通関数
# =========================================================

START_TIME = None


def format_elapsed(seconds):
    h = int(seconds // 3600)
    m = int((seconds % 3600) // 60)
    s = int(seconds % 60)
    return f"{h:02d}:{m:02d}:{s:02d}"


def log(msg):
    elapsed = time.time() - START_TIME
    print(
        f"[{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} "
        f"| +{format_elapsed(elapsed)}] {msg}",
        flush=True,
    )


def log_centrality(mode, msg):
    log(f"[現在の操作指標: {mode}] {msg}")


def input_timeout(prompt, timeout=10, default="y"):
    print(
        f"{prompt} ※{timeout}秒入力がなければ [{default}] で続行します: ",
        end="",
        flush=True,
    )

    start = time.time()
    chars = []

    while time.time() - start < timeout:
        if msvcrt.kbhit():
            ch = msvcrt.getwch()

            if ch in ("\r", "\n"):
                print()
                break

            chars.append(ch)
            print(ch, end="", flush=True)

        time.sleep(0.05)

    else:
        print()
        return default

    s = "".join(chars).strip()
    return s if s else default


def set_seed():
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


def robust_z(s):
    s = pd.to_numeric(s, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
    med = s.median()
    mad = (s - med).abs().median()

    if mad == 0:
        return (s - s.mean()) / (s.std() + 1e-9)

    return 0.6745 * (s - med) / mad


def safe_float(x, default=0.0):
    try:
        x = float(x)
        if np.isfinite(x):
            return x
        return default
    except Exception:
        return default


def percentile_rank_series(s):
    s = (
        pd.to_numeric(s, errors="coerce")
        .replace([np.inf, -np.inf], np.nan)
        .fillna(0)
    )
    return s.rank(pct=True, ascending=True)


def make_edge_ranking(
    edges,
    ids,
    time_steps_np,
    y_np,
    deg,
    bet,
    pr,
    clo,
    hits_hub,
    constraint,
    out_file=EDGE_RANKING_FILE,
    top_file=EDGE_RANKING_TOP_FILE,
    top_n=EDGE_RANKING_TOP_N,
):
    """
    txs_edgelist.txt 由来の各エッジにスコアを付けてランキングする。

    edge_score は以下の合成:
      - edge_structural_score: 70%
      - edge_temporal_score: 20%
      - edge_label_attention: 10%

    constraint は低い値がブリッジ的に重要な場合があるため、
    constraint_min を低いほど高評価に反転して利用する。
    """
    log("エッジランキング作成開始")

    rows = []

    for u_idx, v_idx in edges:
        u_idx = int(u_idx)
        v_idx = int(v_idx)

        u_tx = int(ids[u_idx])
        v_tx = int(ids[v_idx])

        u_time = int(time_steps_np[u_idx])
        v_time = int(time_steps_np[v_idx])

        u_label = int(y_np[u_idx])
        v_label = int(y_np[v_idx])

        u_degree = safe_float(deg.get(u_idx, 0))
        v_degree = safe_float(deg.get(v_idx, 0))

        u_bet = safe_float(bet.get(u_idx, 0))
        v_bet = safe_float(bet.get(v_idx, 0))

        u_pr = safe_float(pr.get(u_idx, 0))
        v_pr = safe_float(pr.get(v_idx, 0))

        u_clo = safe_float(clo.get(u_idx, 0))
        v_clo = safe_float(clo.get(v_idx, 0))

        u_hub = safe_float(hits_hub.get(u_idx, 0))
        v_hub = safe_float(hits_hub.get(v_idx, 0))

        u_constraint = safe_float(constraint.get(u_idx, 0))
        v_constraint = safe_float(constraint.get(v_idx, 0))

        time_gap = abs(u_time - v_time)
        same_time_step = 1 if u_time == v_time else 0

        has_unknown = 1 if (u_label == -1 or v_label == -1) else 0
        has_illicit = 1 if (u_label == 0 or v_label == 0) else 0
        unknown_unknown = 1 if (u_label == -1 and v_label == -1) else 0

        illicit_unknown = 1 if (
            (u_label == 0 and v_label == -1)
            or
            (u_label == -1 and v_label == 0)
        ) else 0

        rows.append({
            "u_txId": u_tx,
            "v_txId": v_tx,
            "u_index": u_idx,
            "v_index": v_idx,
            "u_time_step": u_time,
            "v_time_step": v_time,
            "time_gap": time_gap,
            "same_time_step": same_time_step,
            "u_label": u_label,
            "v_label": v_label,
            "has_unknown": has_unknown,
            "has_illicit": has_illicit,
            "unknown_unknown": unknown_unknown,
            "illicit_unknown": illicit_unknown,
            "u_degree": u_degree,
            "v_degree": v_degree,
            "degree_sum": u_degree + v_degree,
            "degree_max": max(u_degree, v_degree),
            "u_betweenness": u_bet,
            "v_betweenness": v_bet,
            "betweenness_sum": u_bet + v_bet,
            "betweenness_max": max(u_bet, v_bet),
            "u_pagerank": u_pr,
            "v_pagerank": v_pr,
            "pagerank_sum": u_pr + v_pr,
            "pagerank_max": max(u_pr, v_pr),
            "u_closeness": u_clo,
            "v_closeness": v_clo,
            "closeness_sum": u_clo + v_clo,
            "closeness_max": max(u_clo, v_clo),
            "u_hits_hub": u_hub,
            "v_hits_hub": v_hub,
            "hits_hub_sum": u_hub + v_hub,
            "hits_hub_max": max(u_hub, v_hub),
            "u_constraint": u_constraint,
            "v_constraint": v_constraint,
            "constraint_sum": u_constraint + v_constraint,
            "constraint_min": min(u_constraint, v_constraint),
        })

    df_edge = pd.DataFrame(rows)

    if len(df_edge) == 0:
        log("警告: エッジランキング対象が0件です")
        df_edge.to_csv(out_file, index=False, encoding="utf-8-sig")
        return df_edge

    df_edge["degree_sum_pct"] = percentile_rank_series(df_edge["degree_sum"])
    df_edge["betweenness_sum_pct"] = percentile_rank_series(df_edge["betweenness_sum"])
    df_edge["pagerank_sum_pct"] = percentile_rank_series(df_edge["pagerank_sum"])
    df_edge["closeness_sum_pct"] = percentile_rank_series(df_edge["closeness_sum"])
    df_edge["hits_hub_sum_pct"] = percentile_rank_series(df_edge["hits_hub_sum"])

    # constraint は低いほどブリッジ的に重要と見なすため反転する
    df_edge["constraint_bridge_pct"] = (
        1.0 - percentile_rank_series(df_edge["constraint_min"])
    )

    # 時間差は小さいほど近いので反転する
    df_edge["time_gap_close_pct"] = (
        1.0 - percentile_rank_series(df_edge["time_gap"])
    )

    df_edge["same_time_step_pct"] = percentile_rank_series(df_edge["same_time_step"])

    df_edge["edge_structural_score"] = (
        df_edge["degree_sum_pct"]
        + df_edge["betweenness_sum_pct"]
        + df_edge["pagerank_sum_pct"]
        + df_edge["closeness_sum_pct"]
        + df_edge["hits_hub_sum_pct"]
        + df_edge["constraint_bridge_pct"]
    ) / 6.0

    df_edge["edge_temporal_score"] = (
        df_edge["time_gap_close_pct"]
        + df_edge["same_time_step_pct"]
    ) / 2.0

    df_edge["edge_label_attention"] = (
        df_edge["has_unknown"] * 0.5
        + df_edge["has_illicit"] * 0.3
        + df_edge["illicit_unknown"] * 0.2
    )

    df_edge["edge_score"] = (
        df_edge["edge_structural_score"] * 0.7
        + df_edge["edge_temporal_score"] * 0.2
        + df_edge["edge_label_attention"] * 0.1
    )

    df_edge["edge_rank"] = (
        df_edge["edge_score"]
        .rank(ascending=False, method="min")
        .astype(int)
    )

    df_edge = df_edge.sort_values("edge_rank")

    df_edge.to_csv(out_file, index=False, encoding="utf-8-sig")
    log(f"エッジランキングCSV保存: {out_file}")

    if top_n is not None and top_n > 0:
        df_edge.head(top_n).to_csv(top_file, index=False, encoding="utf-8-sig")
        log(f"エッジランキングTop{top_n} CSV保存: {top_file}")

    print("\n==============================")
    print("        エッジランキング Top10")
    print("==============================")

    edge_show_cols = [
        "edge_rank",
        "u_txId",
        "v_txId",
        "u_time_step",
        "v_time_step",
        "time_gap",
        "same_time_step",
        "has_unknown",
        "has_illicit",
        "degree_sum",
        "pagerank_sum",
        "betweenness_sum",
        "constraint_min",
        "edge_structural_score",
        "edge_temporal_score",
        "edge_score",
    ]

    print(
        df_edge.head(10)[edge_show_cols].to_string(
            index=False,
            float_format=lambda x: f"{x:.6f}",
        )
    )

    log("エッジランキング作成完了")
    return df_edge


def prune_edges_by_ranking(df_edge_rank, original_edges, num_nodes,
                           mode=PRUNE_MODE, top_n=PRUNE_TOP_N,
                           keep_ratio=PRUNE_KEEP_RATIO,
                           keep_at_least_one=PRUNE_KEEP_AT_LEAST_ONE_EDGE_PER_NODE,
                           out_file=PRUNED_EDGE_FILE):
    log("枝刈り開始")

    if df_edge_rank is None or len(df_edge_rank) == 0:
        log("警告: edge ranking が空です。枝刈りせず元edgeを使います")
        return original_edges

    if "edge_score" not in df_edge_rank.columns:
        raise ValueError("edge_score列がありません。make_edge_ranking() のスコア計算を確認してください。")

    df_sorted = df_edge_rank.sort_values("edge_score", ascending=False).copy()
    original_edge_count = len(original_edges)

    if mode == "top_n":
        keep_n = int(top_n)
    elif mode == "ratio":
        keep_n = int(original_edge_count * float(keep_ratio))
    else:
        raise ValueError(f"未対応のPRUNE_MODEです: {mode}")

    keep_n = max(1, min(keep_n, len(df_sorted)))
    df_keep = df_sorted.head(keep_n).copy()

    log(
        f"枝刈り設定: mode={mode} "
        f"original_edges={original_edge_count} "
        f"keep_edges_initial={len(df_keep)} "
        f"keep_ratio_actual={len(df_keep)/max(original_edge_count,1):.4f}"
    )

    if keep_at_least_one:
        log("孤立ノード対策開始: 各ノード最低1本のedgeを確保")
        covered = set(df_keep["u_index"].astype(int)) | set(df_keep["v_index"].astype(int))
        missing = set(range(num_nodes)) - covered
        log(f"枝刈り後に孤立しうるノード数={len(missing)}")

        extra = []
        if missing:
            for _, row in df_sorted.iterrows():
                u = int(row["u_index"])
                v = int(row["v_index"])
                if u in missing or v in missing:
                    extra.append(row)
                    missing.discard(u)
                    missing.discard(v)
                if not missing:
                    break

        if extra:
            df_keep = pd.concat([df_keep, pd.DataFrame(extra)], ignore_index=True)
            df_keep = df_keep.drop_duplicates(subset=["u_index", "v_index"], keep="first")

        log(f"孤立ノード対策後 keep_edges={len(df_keep)} 残孤立候補={len(missing)}")

    df_keep = df_keep.sort_values("edge_score", ascending=False).copy()
    pruned_edges = df_keep[["u_index", "v_index"]].astype(int).values.tolist()

    df_keep.to_csv(out_file, index=False, encoding="utf-8-sig")
    log(f"枝刈り後エッジCSV保存: {out_file}")

    reduction_rate = 1.0 - len(pruned_edges) / max(original_edge_count, 1)
    log(
        f"枝刈り完了: original_edges={original_edge_count} "
        f"pruned_edges={len(pruned_edges)} 削減率={reduction_rate:.4f}"
    )

    print("\n==============================")
    print("        枝刈り結果")
    print("==============================")
    print(f"元edge数      : {original_edge_count}")
    print(f"枝刈り後edge数: {len(pruned_edges)}")
    print(f"削減率        : {reduction_rate:.4f}")
    print(f"出力CSV       : {out_file}")

    return pruned_edges


# =========================================================
# 3. GCNモデル
# =========================================================

class GCN(torch.nn.Module):
    def __init__(self, in_dim, hidden_dim, out_dim=2):
        super().__init__()
        self.conv1 = GCNConv(in_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, hidden_dim)
        self.lin = torch.nn.Linear(hidden_dim, out_dim)

    def forward(self, x, edge_index):
        x = self.conv1(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)

        x = self.conv2(x, edge_index)
        x = F.relu(x)
        x = F.dropout(x, p=DROPOUT, training=self.training)

        return self.lin(x)


# =========================================================
# 4. 評価関数
# =========================================================

def tune_threshold(y_true, prob_illicit):
    best_th = 0.5
    best_f1 = -1

    for th in np.arange(0.05, 0.96, 0.05):
        pred = (prob_illicit >= th).astype(int)
        f1 = f1_score(y_true, pred, pos_label=1, zero_division=0)

        if f1 > best_f1:
            best_f1 = f1
            best_th = th

    return best_th, best_f1


def extract_metrics(report_dict):
    return {
        "illicit_precision": report_dict["illicit"]["precision"],
        "illicit_recall": report_dict["illicit"]["recall"],
        "illicit_f1": report_dict["illicit"]["f1-score"],
        "accuracy": report_dict["accuracy"],
        "macro_f1": report_dict["macro avg"]["f1-score"],
        "weighted_f1": report_dict["weighted avg"]["f1-score"],
    }


# =========================================================
# 5. GCN学習
# =========================================================

def run_gcn_training(
    data_base,
    X_raw,
    y_train_np,
    original_y_np,
    time_steps_np,
    device,
    label_name="first",
):
    log(f"GCN学習開始: {label_name}")

    train_mask_np = (
        (y_train_np != -1)
        & (time_steps_np <= TRAIN_END_STEP)
    )

    val_mask_np = (
        (original_y_np != -1)
        & (time_steps_np > TRAIN_END_STEP)
        & (time_steps_np <= VAL_END_STEP)
    )

    test_mask_np = (
        (original_y_np != -1)
        & (time_steps_np > VAL_END_STEP)
    )

    log(
        f"{label_name}: train={train_mask_np.sum()} "
        f"val={val_mask_np.sum()} "
        f"test={test_mask_np.sum()}"
    )

    scaler = StandardScaler()
    X_scaled = X_raw.astype(np.float32).copy()

    X_scaled[train_mask_np] = scaler.fit_transform(X_scaled[train_mask_np])
    X_scaled[~train_mask_np] = scaler.transform(X_scaled[~train_mask_np])

    data = Data(
        x=torch.tensor(X_scaled, dtype=torch.float),
        edge_index=data_base.edge_index.clone(),
        y=torch.tensor(y_train_np, dtype=torch.long),
        time_steps=data_base.time_steps.clone(),
        node_ids=data_base.node_ids.clone(),
    )

    data.train_mask = torch.tensor(train_mask_np, dtype=torch.bool)
    data.val_mask = torch.tensor(val_mask_np, dtype=torch.bool)
    data.test_mask = torch.tensor(test_mask_np, dtype=torch.bool)

    data = data.to(device)

    model = GCN(
        in_dim=data.x.size(1),
        hidden_dim=HIDDEN_DIM,
        out_dim=2,
    ).to(device)

    train_y = data.y[data.train_mask]
    counts = torch.bincount(train_y, minlength=2).float()

    weights = counts.sum() / (counts + 1e-9)
    weights = weights / weights.mean()
    weights = weights.to(device)

    log(f"{label_name}: class counts={counts.detach().cpu().numpy()}")
    log(f"{label_name}: class weights={weights.detach().cpu().numpy()}")

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=LR,
        weight_decay=WEIGHT_DECAY,
    )

    best_val_f1 = -1
    best_state = None

    for epoch in range(1, EPOCHS + 1):
        model.train()
        optimizer.zero_grad()

        out = model(data.x, data.edge_index)

        loss = F.cross_entropy(
            out[data.train_mask],
            data.y[data.train_mask],
            weight=weights,
        )

        loss.backward()
        optimizer.step()

        model.eval()
        with torch.no_grad():
            out = model(data.x, data.edge_index)
            prob = F.softmax(out, dim=1)

            val_prob_illicit = prob[data.val_mask, 0].detach().cpu().numpy()
            val_true_raw = original_y_np[val_mask_np]
            val_true = (val_true_raw == 0).astype(int)

            th, val_f1 = tune_threshold(val_true, val_prob_illicit)

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_state = {
                "model": model.state_dict(),
                "threshold": th,
                "epoch": epoch,
            }

        if epoch == 1 or epoch % LOG_EPOCH_INTERVAL == 0 or epoch == EPOCHS:
            log(
                f"{label_name}: epoch={epoch:03d}/{EPOCHS} "
                f"loss={loss.item():.6f} "
                f"val_illicit_f1={val_f1:.4f} "
                f"best={best_val_f1:.4f} "
                f"th={th:.2f}"
            )

    if best_state is None:
        raise RuntimeError("best_state が作成されませんでした")

    model.load_state_dict(best_state["model"])
    model.eval()

    with torch.no_grad():
        out = model(data.x, data.edge_index)
        prob = F.softmax(out, dim=1)

    test_prob_illicit = prob[data.test_mask, 0].detach().cpu().numpy()
    test_true_raw = original_y_np[test_mask_np]
    test_true = (test_true_raw == 0).astype(int)
    test_pred = (test_prob_illicit >= best_state["threshold"]).astype(int)

    cm = confusion_matrix(test_true, test_pred)

    report_text = classification_report(
        test_true,
        test_pred,
        target_names=["licit", "illicit"],
        zero_division=0,
    )

    report_dict = classification_report(
        test_true,
        test_pred,
        target_names=["licit", "illicit"],
        zero_division=0,
        output_dict=True,
    )

    log(f"GCN学習完了: {label_name}")

    return {
        "prob": prob.detach().cpu().numpy(),
        "best_epoch": best_state["epoch"],
        "best_threshold": best_state["threshold"],
        "best_val_f1": best_val_f1,
        "test_cm": cm,
        "test_report_text": report_text,
        "test_report_dict": report_dict,
        "train_count": int(train_mask_np.sum()),
        "val_count": int(val_mask_np.sum()),
        "test_count": int(test_mask_np.sum()),
    }


# =========================================================
# 6. 指標ランキング作成
# =========================================================

def add_metric_rankings(df_u):
    df = df_u.copy()

    df["degree_score"] = df["d_z"].abs()
    df["betweenness_score"] = df["b_z"].abs()
    df["pagerank_score"] = df["p_z"].abs()
    df["closeness_score"] = df["c_z"].abs()
    df["hits_hub_score"] = df["h_z"].abs()
    df["constraint_score"] = df["constraint_z"].abs()
    df["temporal_burst_score"] = df["tb_z"].abs()
    df["transaction_velocity_score"] = df["tv_z"].abs()

    for mode in METRIC_MODES:
        df[f"{mode}_rank"] = df[f"{mode}_score"].rank(
            ascending=False,
            method="min",
        ).astype(int)

    return df


# 旧関数名との互換性
add_centrality_rankings = add_metric_rankings


def get_metric_score_column(mode):
    mapping = {
        "degree": "degree_score",
        "betweenness": "betweenness_score",
        "pagerank": "pagerank_score",
        "closeness": "closeness_score",
        "hits_hub": "hits_hub_score",
        "constraint": "constraint_score",
        "temporal_burst": "temporal_burst_score",
        "transaction_velocity": "transaction_velocity_score",
    }

    if mode not in mapping:
        raise ValueError(f"未対応の指標モードです: {mode}")

    return mapping[mode]


# 旧関数名との互換性
get_centrality_score_column = get_metric_score_column


def get_metric_rank_column(mode):
    mapping = {
        "degree": "degree_rank",
        "betweenness": "betweenness_rank",
        "pagerank": "pagerank_rank",
        "closeness": "closeness_rank",
        "hits_hub": "hits_hub_rank",
        "constraint": "constraint_rank",
        "temporal_burst": "temporal_burst_rank",
        "transaction_velocity": "transaction_velocity_rank",
    }

    if mode not in mapping:
        raise ValueError(f"未対応の指標モードです: {mode}")

    return mapping[mode]


# 旧関数名との互換性
get_centrality_rank_column = get_metric_rank_column


# =========================================================
# 7. GCNスコアとの合成
# =========================================================

def add_gcn_and_combined_score(df_u, prob_all, centrality_mode, suffix=""):
    df = df_u.copy()

    score_col = get_metric_score_column(centrality_mode)
    prob_illicit_all = prob_all[:, 0]

    col_prob = f"gcn_illicit_prob{suffix}"
    col_combined = f"combined_score{suffix}"

    df[col_prob] = df["node_index"].map(
        lambda idx: float(prob_illicit_all[int(idx)])
    )

    df[col_combined] = (
        df[score_col].rank(pct=True) * 0.5
        + df[col_prob].rank(pct=True) * 0.5
    )

    return df


def make_summary(df, centrality_mode, prob_col, score_col):
    selected_score_col = get_metric_score_column(centrality_mode)

    return df.groupby("time_step").agg(
        unknown_count=(score_col, "count"),
        avg_degree=("degree", "mean"),
        avg_betweenness=("betweenness", "mean"),
        avg_pagerank=("pagerank", "mean"),
        avg_closeness=("closeness", "mean"),
        avg_hits_hub=("hits_hub", "mean"),
        avg_constraint=("constraint", "mean"),
        avg_temporal_burst=("temporal_burst", "mean"),
        avg_transaction_velocity=("transaction_velocity", "mean"),
        max_selected_metric_score=(selected_score_col, "max"),
        avg_selected_metric_score=(selected_score_col, "mean"),
        avg_gcn_illicit_prob=(prob_col, "mean"),
        max_gcn_illicit_prob=(prob_col, "max"),
        avg_combined_score=(score_col, "mean"),
        max_combined_score=(score_col, "max"),
    ).reset_index()


# =========================================================
# 8. プロット保存
# =========================================================

def plot_final_table(df_final_show, centrality_mode):
    log_centrality(centrality_mode, "最終比較表プロット作成開始")

    plot_df = df_final_show.copy()

    plt.figure(figsize=PLOT_FIGSIZE)

    plt.plot(plot_df["条件"], plot_df["F1"], marker="o", label="F1")
    plt.plot(plot_df["条件"], plot_df["再現率(R)"], marker="o", label="再現率(R)")
    plt.plot(plot_df["条件"], plot_df["適合率(P)"], marker="o", label="適合率(P)")
    plt.plot(plot_df["条件"], plot_df["正解率"], marker="o", label="正解率")

    plt.title(f"Pseudo Illicit Top数別 GCN性能比較: {centrality_mode}")
    plt.xlabel("条件")
    plt.ylabel("スコア")
    plt.xticks(rotation=45)
    plt.grid(True)
    plt.legend()
    plt.tight_layout()

    out_png = f"pseudo_illicit_final_plot_{centrality_mode}.png"
    plt.savefig(out_png, dpi=PLOT_DPI)

    log_centrality(centrality_mode, f"PNG保存: {out_png}")

    plt.close()


def show_all_pngs():
    log("保存済みPNG一覧を表示します")

    png_files = sorted(
        Path(".").glob("pseudo_illicit_final_plot_*.png")
    )

    log(f"PNG数: {len(png_files)}")

    for i, png_file in enumerate(png_files, start=1):
        log(f"PNG表示 ({i}/{len(png_files)}): {png_file.name}")

        img = mpimg.imread(png_file)

        plt.figure(figsize=(16, 9))
        plt.imshow(img)
        plt.axis("off")
        plt.title(png_file.name, fontsize=16)
        plt.tight_layout()
        plt.show()
        plt.close()


# =========================================================
# 9. 指標モード別 実験処理
# =========================================================

def run_experiment_for_centrality_mode(
    centrality_mode,
    df_u,
    result_first,
    data_base,
    X_raw,
    y_np,
    original_y_np,
    time_steps_np,
    device,
):
    log("==================================================")
    log_centrality(centrality_mode, "指標モード別実験開始")
    log("==================================================")

    score_col = get_metric_score_column(centrality_mode)
    rank_col = get_metric_rank_column(centrality_mode)

    log_centrality(centrality_mode, "ランキングCSV作成中")

    df_rank_file = df_u.sort_values(rank_col).copy()
    rank_file = f"unknown_outlier_{centrality_mode}_ranking.csv"
    df_rank_file.to_csv(rank_file, index=False, encoding="utf-8-sig")

    log_centrality(centrality_mode, f"CSV保存: {rank_file}")

    log_centrality(centrality_mode, "1回目GCNスコアとの合成開始")

    df_first = add_gcn_and_combined_score(
        df_u=df_u,
        prob_all=result_first["prob"],
        centrality_mode=centrality_mode,
        suffix=f"_first_{centrality_mode}",
    )

    prob_col_first = f"gcn_illicit_prob_first_{centrality_mode}"
    combined_col_first = f"combined_score_first_{centrality_mode}"

    detail_first_file = f"unknown_outlier_with_gcn_score_first_{centrality_mode}.csv"
    summary_first_file = f"unknown_outlier_summary_by_time_step_first_{centrality_mode}.csv"

    summary_first = make_summary(
        df_first,
        centrality_mode=centrality_mode,
        prob_col=prob_col_first,
        score_col=combined_col_first,
    )

    df_first.to_csv(detail_first_file, index=False, encoding="utf-8-sig")
    summary_first.to_csv(summary_first_file, index=False, encoding="utf-8-sig")

    log_centrality(centrality_mode, f"CSV保存: {detail_first_file}")
    log_centrality(centrality_mode, f"CSV保存: {summary_first_file}")

    compare_rows = []
    detailed_results = {}

    total_patterns = len(PSEUDO_LIST)

    for pattern_i, top_n in enumerate(PSEUDO_LIST, start=1):
        remain = total_patterns - pattern_i

        log_centrality(
            centrality_mode,
            f"pseudo illicit Top{top_n} 開始 "
            f"({pattern_i}/{total_patterns}) 残り={remain}",
        )

        log_centrality(
            centrality_mode,
            f"Top{top_n}: {score_col} に基づいてランキング抽出中",
        )

        df_ranked = df_first.sort_values(
            score_col,
            ascending=False,
        ).copy()

        if PSEUDO_ONLY_TRAIN_PERIOD:
            df_ranked = df_ranked[df_ranked["time_step"] <= TRAIN_END_STEP]
            log_centrality(
                centrality_mode,
                f"Top{top_n}: train期間内 unknown のみに制限",
            )

        df_pseudo = df_ranked.head(top_n).copy()
        pseudo_indices = df_pseudo["node_index"].astype(int).values

        log_centrality(
            centrality_mode,
            f"Top{top_n}: pseudo illicit 数={len(pseudo_indices)}",
        )

        df_pseudo["metric_mode"] = centrality_mode
        df_pseudo["centrality_mode"] = centrality_mode
        df_pseudo["pseudo_label"] = "illicit"
        df_pseudo["pseudo_label_value"] = 0
        df_pseudo["pseudo_source"] = f"{centrality_mode}_rank_top{top_n}"

        pseudo_file = f"pseudo_illicit_{centrality_mode}_rank_top{top_n}.csv"
        df_pseudo.to_csv(pseudo_file, index=False, encoding="utf-8-sig")

        log_centrality(centrality_mode, f"CSV保存: {pseudo_file}")

        y_np_recalc = y_np.copy()
        y_np_recalc[pseudo_indices] = 0

        log_centrality(
            centrality_mode,
            f"Top{top_n}: 疑似ラベル反映完了。GCN再学習開始",
        )

        result = run_gcn_training(
            data_base=data_base,
            X_raw=X_raw,
            y_train_np=y_np_recalc,
            original_y_np=original_y_np,
            time_steps_np=time_steps_np,
            device=device,
            label_name=f"{centrality_mode}_top{top_n}",
        )

        log_centrality(
            centrality_mode,
            f"Top{top_n}: GCN再学習完了。スコア再計算開始",
        )

        df_recalc = add_gcn_and_combined_score(
            df_u=df_u,
            prob_all=result["prob"],
            centrality_mode=centrality_mode,
            suffix=f"_top{top_n}_{centrality_mode}",
        )

        prob_col_recalc = f"gcn_illicit_prob_top{top_n}_{centrality_mode}"
        combined_col_recalc = f"combined_score_top{top_n}_{centrality_mode}"

        df_recalc[prob_col_first] = df_first[prob_col_first]
        df_recalc[combined_col_first] = df_first[combined_col_first]
        df_recalc["metric_mode"] = centrality_mode
        df_recalc["centrality_mode"] = centrality_mode
        df_recalc["pseudo_illicit_used"] = df_recalc["node_index"].isin(pseudo_indices)

        detail_file = f"unknown_outlier_with_gcn_score_top{top_n}_{centrality_mode}.csv"
        summary_file = f"unknown_outlier_summary_by_time_step_top{top_n}_{centrality_mode}.csv"

        df_recalc.to_csv(detail_file, index=False, encoding="utf-8-sig")

        summary_recalc = make_summary(
            df_recalc,
            centrality_mode=centrality_mode,
            prob_col=prob_col_recalc,
            score_col=combined_col_recalc,
        )

        summary_recalc.to_csv(summary_file, index=False, encoding="utf-8-sig")

        log_centrality(centrality_mode, f"CSV保存: {detail_file}")
        log_centrality(centrality_mode, f"CSV保存: {summary_file}")

        metrics = extract_metrics(result["test_report_dict"])
        cm = result["test_cm"]

        compare_rows.append({
            "metric_mode": centrality_mode,
            "centrality_mode": centrality_mode,
            "mode": f"{centrality_mode}_pseudo_top{top_n}",
            "pseudo_top_n": top_n,
            "pseudo_count": len(pseudo_indices),
            "train_count": result["train_count"],
            "val_count": result["val_count"],
            "test_count": result["test_count"],
            "best_epoch": result["best_epoch"],
            "best_threshold": result["best_threshold"],
            "best_val_f1": result["best_val_f1"],
            "tn_licit_correct": int(cm[0, 0]),
            "fp_licit_to_illicit": int(cm[0, 1]),
            "fn_illicit_to_licit": int(cm[1, 0]),
            "tp_illicit_correct": int(cm[1, 1]),
            **metrics,
        })

        detailed_results[top_n] = result

        log_centrality(
            centrality_mode,
            f"pseudo illicit Top{top_n} 完了 "
            f"({pattern_i}/{total_patterns})",
        )

    cm_first = result_first["test_cm"]
    metrics_first = extract_metrics(result_first["test_report_dict"])

    first_row = {
        "metric_mode": centrality_mode,
        "centrality_mode": centrality_mode,
        "mode": f"{centrality_mode}_first_no_pseudo",
        "pseudo_top_n": 0,
        "pseudo_count": 0,
        "train_count": result_first["train_count"],
        "val_count": result_first["val_count"],
        "test_count": result_first["test_count"],
        "best_epoch": result_first["best_epoch"],
        "best_threshold": result_first["best_threshold"],
        "best_val_f1": result_first["best_val_f1"],
        "tn_licit_correct": int(cm_first[0, 0]),
        "fp_licit_to_illicit": int(cm_first[0, 1]),
        "fn_illicit_to_licit": int(cm_first[1, 0]),
        "tp_illicit_correct": int(cm_first[1, 1]),
        **metrics_first,
    }

    df_compare = pd.DataFrame([first_row] + compare_rows)

    compare_file = f"pseudo_illicit_compare_summary_{centrality_mode}.csv"
    df_compare.to_csv(compare_file, index=False, encoding="utf-8-sig")

    log_centrality(centrality_mode, f"CSV保存: {compare_file}")

    print("\n==============================")
    print(f"        最終比較表: {centrality_mode}")
    print("==============================")

    df_final = df_compare.copy()

    order = (
        [f"{centrality_mode}_first_no_pseudo"]
        + [f"{centrality_mode}_pseudo_top{i}" for i in PSEUDO_LIST]
    )

    df_final["mode"] = pd.Categorical(
        df_final["mode"],
        categories=order,
        ordered=True,
    )

    df_final = df_final.sort_values("mode")

    df_final_show = pd.DataFrame({
        "指標": df_final["metric_mode"],
        "条件": [
            "1回目" if int(n) == 0 else f"Top{int(n)}"
            for n in df_final["pseudo_top_n"]
        ],
        "pseudo数": df_final["pseudo_count"],
        "閾値": df_final["best_threshold"],
        "検証F1": df_final["best_val_f1"],
        "適合率(P)": df_final["illicit_precision"],
        "再現率(R)": df_final["illicit_recall"],
        "F1": df_final["illicit_f1"],
        "正解率": df_final["accuracy"],
        "誤検出(FP)": df_final["fp_licit_to_illicit"],
        "見逃し(FN)": df_final["fn_illicit_to_licit"],
        "検出(TP)": df_final["tp_illicit_correct"],
    })

    print(
        df_final_show.to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}",
        )
    )

    final_table_file = f"pseudo_illicit_final_table_{centrality_mode}.csv"
    df_final_show.to_csv(final_table_file, index=False, encoding="utf-8-sig")

    log_centrality(centrality_mode, f"CSV保存: {final_table_file}")

    plot_final_table(df_final_show.rename(columns={"指標": "中心性"}), centrality_mode)

    best_f1_row = df_compare.sort_values(
        "illicit_f1",
        ascending=False,
    ).iloc[0]

    print("\n==============================")
    print(f"        最良モデル F1基準: {centrality_mode}")
    print("==============================")
    print(f"条件       : {best_f1_row['mode']}")
    print(f"pseudo数   : {best_f1_row['pseudo_count']}")
    print(f"illicit F1 : {best_f1_row['illicit_f1']:.4f}")
    print(f"再現率     : {best_f1_row['illicit_recall']:.4f}")
    print(f"適合率     : {best_f1_row['illicit_precision']:.4f}")
    print(f"閾値       : {best_f1_row['best_threshold']:.2f}")

    log_centrality(centrality_mode, "指標モード別実験完了")

    return df_compare


# =========================================================
# 10. main
# =========================================================

def main():
    global START_TIME
    START_TIME = time.time()

    setup_japanese_font()

    log("START")
    set_seed()

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    log(f"device={device}")

    log("CSV読み込み開始")

    df_f = pd.read_csv(TXS_FEATURES)
    df_c = pd.read_csv(TXS_CLASSES)
    df_e = pd.read_csv(TXS_EDGES)

    log(f"features={df_f.shape} classes={df_c.shape} edges={df_e.shape}")

    id_col_f = df_f.columns[0]
    id_col_c = df_c.columns[0]

    df = df_f.merge(df_c, left_on=id_col_f, right_on=id_col_c, how="left")

    if df["class"].isna().any():
        log("警告: class欠損があります。unknown扱いにします")
        df["class"] = df["class"].fillna(3)

    ids = df[id_col_f].astype(int).values
    id2idx = {int(v): i for i, v in enumerate(ids)}

    y_np = df["class"].map(
        lambda x: 0 if int(x) == 1 else 1 if int(x) == 2 else -1
    ).values

    original_y_np = y_np.copy()
    time_steps_np = df["Time step"].astype(int).values

    log(
        f"illicit={(y_np == 0).sum()} "
        f"licit={(y_np == 1).sum()} "
        f"unknown={(y_np == -1).sum()}"
    )

    exclude_cols = {
        id_col_f,
        id_col_c,
        "class",
        "Time step",
    }

    feature_cols = [c for c in df.columns if c not in exclude_cols]
    log(f"特徴量数={len(feature_cols)}")

    X = df[feature_cols].apply(pd.to_numeric, errors="coerce")
    X = X.replace([np.inf, -np.inf], np.nan).fillna(0)
    X_raw = X.values.astype(np.float32)

    log("edge構築開始")

    col0, col1 = df_e.columns[:2]

    edges = []
    skipped = 0

    for u, v in zip(df_e[col0], df_e[col1]):
        try:
            u = int(u)
            v = int(v)
        except Exception:
            skipped += 1
            continue

        if u in id2idx and v in id2idx:
            edges.append([id2idx[u], id2idx[v]])
        else:
            skipped += 1

    log(f"edge構築完了: edges={len(edges)} skipped={skipped}")

    # =====================================================
    # 1) 全エッジで仮グラフを作り、エッジランキングを作成
    # =====================================================

    log("全エッジ版 edge_index 構築開始")

    edge_index_full = torch.tensor(edges, dtype=torch.long).T
    edge_index_full = to_undirected(edge_index_full)
    edge_index_full, _ = add_self_loops(edge_index_full, num_nodes=len(ids))

    data_full = Data(
        x=torch.tensor(X_raw, dtype=torch.float),
        edge_index=edge_index_full,
        y=torch.tensor(y_np, dtype=torch.long),
        time_steps=torch.tensor(time_steps_np, dtype=torch.long),
        node_ids=torch.tensor(ids, dtype=torch.long),
    )

    log("全エッジ版グラフ指標計算開始")

    G_full = to_networkx(data_full, to_undirected=True)
    G_full.remove_edges_from(nx.selfloop_edges(G_full))

    log(f"Full Graph nodes={G_full.number_of_nodes()} edges={G_full.number_of_edges()}")

    log_centrality("degree_full", "degree計算開始")
    deg_full = dict(G_full.degree())
    log_centrality("degree_full", "degree計算完了")

    log_centrality("betweenness_full", "betweenness計算開始")
    bet_full = nx.betweenness_centrality(G_full, k=BETWEENNESS_K, seed=SEED)
    log_centrality("betweenness_full", "betweenness計算完了")

    log_centrality("pagerank_full", "pagerank計算開始")
    pr_full = nx.pagerank(G_full)
    log_centrality("pagerank_full", "pagerank計算完了")

    log_centrality("closeness_full", "closeness計算開始")
    clo_full = nx.closeness_centrality(G_full)
    log_centrality("closeness_full", "closeness計算完了")

    log_centrality("hits_hub_full", "HITS Hub計算開始")
    try:
        hits_hub_full, hits_auth_full = nx.hits(G_full, max_iter=HITS_MAX_ITER, tol=HITS_TOL, normalized=True)
    except Exception as e:
        log_centrality("hits_hub_full", f"HITS Hub計算失敗: {e}")
        hits_hub_full = {n: 0.0 for n in G_full.nodes()}
    log_centrality("hits_hub_full", "HITS Hub計算完了")

    log_centrality("constraint_full", "constraint計算開始")
    try:
        constraint_full = nx.constraint(G_full)
    except Exception as e:
        log_centrality("constraint_full", f"constraint計算失敗: {e}")
        constraint_full = {n: 0.0 for n in G_full.nodes()}
    log_centrality("constraint_full", "constraint計算完了")

    log("全エッジ版グラフ指標計算完了")

    df_edge_rank = None
    if ENABLE_EDGE_RANKING or ENABLE_EDGE_PRUNING:
        df_edge_rank = make_edge_ranking(
            edges=edges,
            ids=ids,
            time_steps_np=time_steps_np,
            y_np=y_np,
            deg=deg_full,
            bet=bet_full,
            pr=pr_full,
            clo=clo_full,
            hits_hub=hits_hub_full,
            constraint=constraint_full,
        )

        # =================================================
        # エッジランキング直後の確認
        # =================================================
        ans_edge = input_timeout(
            "\nエッジランキング完了。枝刈り・GCNへ進みますか？ [y/N]",
            timeout=INPUT_TIMEOUT_SEC,
            default=INPUT_DEFAULT,
        ).strip().lower()

        if ans_edge not in ["y", "yes"]:
            log("ユーザー操作により、枝刈り・GCNを実行せず終了します")

            total_elapsed = time.time() - START_TIME

            print("\n==============================")
            print("        総処理時間")
            print("==============================")
            print(format_elapsed(total_elapsed))

            log("DONE")
            return

    # =====================================================
    # 2) edge_score上位だけを残して、実際に枝刈りする
    # =====================================================

    if ENABLE_EDGE_PRUNING:
        edges_for_gcn = prune_edges_by_ranking(
            df_edge_rank=df_edge_rank,
            original_edges=edges,
            num_nodes=len(ids),
        )
    else:
        log("枝刈り無効: 全edgeをGCNに使用します")
        edges_for_gcn = edges

    # =====================================================
    # 3) 枝刈り後グラフで edge_index を作り、中心性とGCNを実行
    # =====================================================

    log("GCN用 edge_index 構築開始")

    edge_index = torch.tensor(edges_for_gcn, dtype=torch.long).T
    edge_index = to_undirected(edge_index)
    edge_index, _ = add_self_loops(edge_index, num_nodes=len(ids))

    data_base = Data(
        x=torch.tensor(X_raw, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y_np, dtype=torch.long),
        time_steps=torch.tensor(time_steps_np, dtype=torch.long),
        node_ids=torch.tensor(ids, dtype=torch.long),
    )

    log("枝刈り後グラフ指標計算開始")

    G = to_networkx(data_base, to_undirected=True)
    G.remove_edges_from(nx.selfloop_edges(G))

    log(f"Pruned Graph nodes={G.number_of_nodes()} edges={G.number_of_edges()}")

    log_centrality("degree", "degree計算開始")
    deg = dict(G.degree())
    log_centrality("degree", "degree計算完了")

    log_centrality("betweenness", "betweenness計算開始")
    bet = nx.betweenness_centrality(G, k=BETWEENNESS_K, seed=SEED)
    log_centrality("betweenness", "betweenness計算完了")

    log_centrality("pagerank", "pagerank計算開始")
    pr = nx.pagerank(G)
    log_centrality("pagerank", "pagerank計算完了")

    log_centrality("closeness", "closeness計算開始")
    clo = nx.closeness_centrality(G)
    log_centrality("closeness", "closeness計算完了")

    log_centrality("hits_hub", "HITS Hub計算開始")
    try:
        hits_hub, hits_auth = nx.hits(G, max_iter=HITS_MAX_ITER, tol=HITS_TOL, normalized=True)
    except Exception as e:
        log_centrality("hits_hub", f"HITS Hub計算失敗: {e}")
        hits_hub = {n: 0.0 for n in G.nodes()}
    log_centrality("hits_hub", "HITS Hub計算完了")

    log_centrality("constraint", "constraint計算開始")
    try:
        constraint = nx.constraint(G)
    except Exception as e:
        log_centrality("constraint", f"constraint計算失敗: {e}")
        constraint = {n: 0.0 for n in G.nodes()}
    log_centrality("constraint", "constraint計算完了")

    log("枝刈り後グラフ指標計算完了")

    rows = []

    for i in range(len(y_np)):
        if y_np[i] != -1:
            continue

        rows.append({
            "txId": int(ids[i]),
            "node_index": int(i),
            "time_step": int(time_steps_np[i]),
            "degree": safe_float(deg.get(i, 0)),
            "betweenness": safe_float(bet.get(i, 0)),
            "pagerank": safe_float(pr.get(i, 0)),
            "closeness": safe_float(clo.get(i, 0)),
            "hits_hub": safe_float(hits_hub.get(i, 0)),
            "constraint": safe_float(constraint.get(i, 0)),
        })

    df_u = pd.DataFrame(rows)
    log(f"unknown数={len(df_u)}")

    log("時間異常スコア計算開始")

    time_count = df_u["time_step"].value_counts()

    # Temporal Burst:
    # 同じ Time step に unknown ノードがどれだけ集中しているか。
    df_u["temporal_burst"] = df_u["time_step"].map(time_count).astype(float)

    # Transaction Velocity:
    # ここでは簡易近似として、時間ステップあたりの次数を使用。
    # Time step が小さいほど値が大きくなりすぎないよう +1 する。
    df_u["transaction_velocity"] = (
        df_u["degree"].astype(float)
        / (df_u["time_step"].astype(float) + 1.0)
    )

    df_u["temporal_burst"] = df_u["temporal_burst"].replace([np.inf, -np.inf], np.nan).fillna(0)
    df_u["transaction_velocity"] = df_u["transaction_velocity"].replace([np.inf, -np.inf], np.nan).fillna(0)

    log("時間異常スコア計算完了")

    log("8指標のロバストZスコア計算開始")

    df_u["d_z"] = robust_z(df_u["degree"])
    df_u["b_z"] = robust_z(df_u["betweenness"])
    df_u["p_z"] = robust_z(df_u["pagerank"])
    df_u["c_z"] = robust_z(df_u["closeness"])
    df_u["h_z"] = robust_z(df_u["hits_hub"])
    df_u["constraint_z"] = robust_z(df_u["constraint"])
    df_u["tb_z"] = robust_z(df_u["temporal_burst"])
    df_u["tv_z"] = robust_z(df_u["transaction_velocity"])

    log("8指標のロバストZスコア計算完了")

    log("8指標ごとのランキング作成開始")
    df_u = add_metric_rankings(df_u)
    log("8指標ごとのランキング作成完了")

    df_u.to_csv(OUTPUT_CENTER_ALL, index=False, encoding="utf-8-sig")
    log(f"CSV保存: {OUTPUT_CENTER_ALL}")

    for mode in METRIC_MODES:
        rank_col = get_metric_rank_column(mode)
        out_file = f"unknown_outlier_{mode}_ranking.csv"

        log_centrality(mode, "個別ランキングCSV作成中")

        df_u.sort_values(rank_col).to_csv(
            out_file,
            index=False,
            encoding="utf-8-sig",
        )

        log_centrality(mode, f"CSV保存: {out_file}")

    print("\n==============================")
    print("        8指標ランキング概要")
    print("==============================")

    show_cols_base = [
        "txId",
        "node_index",
        "time_step",
        "degree",
        "betweenness",
        "pagerank",
        "closeness",
        "hits_hub",
        "constraint",
        "temporal_burst",
        "transaction_velocity",
    ]

    for mode in METRIC_MODES:
        score_col = get_metric_score_column(mode)
        rank_col = get_metric_rank_column(mode)

        print(f"\n--- {mode} Top10 ---")
        print(
            df_u.sort_values(rank_col)
            .head(10)[show_cols_base + [score_col, rank_col]]
            .to_string(index=False)
        )

    ans = input_timeout(
        "\nGCN学習と疑似ラベル実験を実行しますか？ [y/N]",
        timeout=INPUT_TIMEOUT_SEC,
        default=INPUT_DEFAULT,
    ).strip().lower()

    if ans not in ["y", "yes"]:
        log("GCN学習をスキップします")

        show_all_pngs()

        total_elapsed = time.time() - START_TIME

        print("\n==============================")
        print("        総処理時間")
        print("==============================")
        print(format_elapsed(total_elapsed))

        log("DONE")
        return

    log("1回目のGCN学習開始")

    result_first = run_gcn_training(
        data_base=data_base,
        X_raw=X_raw,
        y_train_np=y_np,
        original_y_np=original_y_np,
        time_steps_np=time_steps_np,
        device=device,
        label_name="first_no_pseudo",
    )

    log("1回目のGCN学習完了")

    all_compare = []

    for metric_mode in METRIC_MODES:
        log_centrality(
            metric_mode,
            "この指標を使った疑似ラベル実験へ移行します",
        )

        df_compare_mode = run_experiment_for_centrality_mode(
            centrality_mode=metric_mode,
            df_u=df_u,
            result_first=result_first,
            data_base=data_base,
            X_raw=X_raw,
            y_np=y_np,
            original_y_np=original_y_np,
            time_steps_np=time_steps_np,
            device=device,
        )

        all_compare.append(df_compare_mode)

    df_all_compare = pd.concat(all_compare, ignore_index=True)

    all_compare_file = "pseudo_illicit_compare_summary_ALL_METRICS.csv"
    df_all_compare.to_csv(
        all_compare_file,
        index=False,
        encoding="utf-8-sig",
    )

    # 旧ファイル名との互換保存
    all_compare_file_compat = "pseudo_illicit_compare_summary_ALL_CENTRALITIES.csv"
    df_all_compare.to_csv(
        all_compare_file_compat,
        index=False,
        encoding="utf-8-sig",
    )

    log(f"CSV保存: {all_compare_file}")
    log(f"CSV保存: {all_compare_file_compat}")

    print("\n==============================")
    print("        8指標 総合比較表")
    print("==============================")

    df_all_show = pd.DataFrame({
        "指標": df_all_compare["metric_mode"],
        "条件": [
            "1回目" if int(n) == 0 else f"Top{int(n)}"
            for n in df_all_compare["pseudo_top_n"]
        ],
        "pseudo数": df_all_compare["pseudo_count"],
        "閾値": df_all_compare["best_threshold"],
        "検証F1": df_all_compare["best_val_f1"],
        "適合率(P)": df_all_compare["illicit_precision"],
        "再現率(R)": df_all_compare["illicit_recall"],
        "F1": df_all_compare["illicit_f1"],
        "正解率": df_all_compare["accuracy"],
        "誤検出(FP)": df_all_compare["fp_licit_to_illicit"],
        "見逃し(FN)": df_all_compare["fn_illicit_to_licit"],
        "検出(TP)": df_all_compare["tp_illicit_correct"],
    })

    print(
        df_all_show.to_string(
            index=False,
            float_format=lambda x: f"{x:.2f}",
        )
    )

    all_final_file = "pseudo_illicit_final_table_ALL_METRICS.csv"
    df_all_show.to_csv(
        all_final_file,
        index=False,
        encoding="utf-8-sig",
    )

    # 旧ファイル名との互換保存
    all_final_file_compat = "pseudo_illicit_final_table_ALL_CENTRALITIES.csv"
    df_all_show.to_csv(
        all_final_file_compat,
        index=False,
        encoding="utf-8-sig",
    )

    log(f"CSV保存: {all_final_file}")
    log(f"CSV保存: {all_final_file_compat}")

    best_f1_row = df_all_compare.sort_values(
        "illicit_f1",
        ascending=False,
    ).iloc[0]

    print("\n==============================")
    print("        全指標 最良モデル F1基準")
    print("==============================")
    print(f"指標       : {best_f1_row['metric_mode']}")
    print(f"条件       : {best_f1_row['mode']}")
    print(f"pseudo数   : {best_f1_row['pseudo_count']}")
    print(f"illicit F1 : {best_f1_row['illicit_f1']:.4f}")
    print(f"再現率     : {best_f1_row['illicit_recall']:.4f}")
    print(f"適合率     : {best_f1_row['illicit_precision']:.4f}")
    print(f"閾値       : {best_f1_row['best_threshold']:.2f}")

    best_recall_row = df_all_compare.sort_values(
        ["illicit_recall", "illicit_f1"],
        ascending=False,
    ).iloc[0]

    print("\n==============================")
    print("        全指標 最良モデル 再現率基準")
    print("==============================")
    print(f"指標       : {best_recall_row['metric_mode']}")
    print(f"条件       : {best_recall_row['mode']}")
    print(f"pseudo数   : {best_recall_row['pseudo_count']}")
    print(f"illicit F1 : {best_recall_row['illicit_f1']:.4f}")
    print(f"再現率     : {best_recall_row['illicit_recall']:.4f}")
    print(f"適合率     : {best_recall_row['illicit_precision']:.4f}")
    print(f"閾値       : {best_recall_row['best_threshold']:.2f}")

    print("\n※ 誤検出(FP) = licit を illicit と誤判定")
    print("※ 見逃し(FN) = illicit を licit と誤判定")
    print("※ 検出(TP) = illicit を正しく illicit と判定")
    print("※ temporal_burst は、同一 Time step 内の unknown ノード集中度です")
    print("※ transaction_velocity は、degree / (Time step + 1) の簡易近似です")
    print("※ constraint_score は constraint のロバストZスコア絶対値です")
    print("※ constraint は低い値がブリッジ的ノードを示す場合もあります")

    show_all_pngs()

    total_elapsed = time.time() - START_TIME

    print("\n==============================")
    print("        総処理時間")
    print("==============================")
    print(format_elapsed(total_elapsed))

    log("DONE")


if __name__ == "__main__":
    main()
